# FedHealth — Notebook 03: Train Model A (Hybrid ANN -> XGBoost)
Trains the CBC multi-condition model **centrally** as the reference model. The ANN is the component that later gets federated (FedAvg + Paillier, Modules 4-5); XGBoost is the local meta-head. Output is a **calibrated per-condition confidence vector** — never a hard label.

In [ ]:
# --- Setup ------------------------------------------------------------------
# Module 3 Colab setup.
#
# Prerequisite:
# Clone the GitHub testing branch into /content/FedHealth-FYP before running
# these notebooks. Dataset CSVs remain in Google Drive at MyDrive/FYP.
#
# This repository stores shared files flat in "Module 3 ML/". The code below
# creates the temporary "fedhealth_ml" package layout expected by the notebooks.

from pathlib import Path
from google.colab import drive
import os
import shutil
import sys

REPO_DIR = Path('/content/FedHealth-FYP')
SOURCE_DIR = REPO_DIR / 'Module 3 ML'
assert SOURCE_DIR.exists(), (
    'Repository files are missing. Clone the GitHub testing branch first.'
)

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

!pip -q install -r "/content/FedHealth-FYP/Module 3 ML/requirements.txt"

RUNTIME_DIR = Path('/content/fedhealth_runtime')
PACKAGE_DIR = RUNTIME_DIR / 'fedhealth_ml'
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

for filename in ['schema.py', 'preprocess.py', 'models.py', 'rules.py']:
    shutil.copy2(SOURCE_DIR / filename, PACKAGE_DIR / filename)

(PACKAGE_DIR / '__init__.py').write_text(
    '# Temporary Colab package for Module 3 validation.\n'
)

if str(RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(RUNTIME_DIR))

DATA_DIR = '/content/drive/MyDrive/FYP'
ANEMIA = f'{DATA_DIR}/anemia_diagnosed_cbc_data_v4.csv'
THAL   = f'{DATA_DIR}/thalassemia_alphanorm.csv'
LEUK   = f'{DATA_DIR}/biased_leukemia_dataset.csv'

for dataset_path in [ANEMIA, THAL, LEUK]:
    assert os.path.exists(dataset_path), f'Dataset not found: {dataset_path}'

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from fedhealth_ml import schema, preprocess as pp

print('fedhealth_ml loaded. Model A features:', len(schema.MODEL_A_FEATURES))
print('All three datasets found in Google Drive.')


## 1. Load Notebook 02 artifacts from local storage or Drive

In [ ]:
ART = '/content/artifacts/modelA'
DRIVE_ART = f'{DATA_DIR}/validation/module3/modelA'
os.makedirs(ART, exist_ok=True)
for filename in ['arrays.npz', 'preprocessor.joblib']:
    local_path = Path(ART) / filename
    drive_path = Path(DRIVE_ART) / filename
    if not local_path.exists():
        assert drive_path.exists(), f'Missing {filename}. Run Notebook 02 first.'
        shutil.copy2(drive_path, local_path)
print('Notebook 02 artifacts ready ->', ART)
d = np.load(f'{ART}/arrays.npz')
Xtr, Xval, Xte = d['Xtr'], d['Xval'], d['Xte']
ytr, yval, yte = d['ytr'], d['yval'], d['yte']
from fedhealth_ml import models, schema
classes = schema.MODEL_A_CLASSES; n = len(classes)
print('train', Xtr.shape, '| classes', n)

## 2. Build + train the ANN (with balanced class weights)

In [ ]:
ann = models.build_ann(input_dim=Xtr.shape[1], n_classes=n, hidden=(64,32), dropout=0.3)
ann.summary()
cw = models.class_weights(ytr, n)
hist = models.train_ann(ann, Xtr, ytr, Xval, yval, class_weight=cw,
                        epochs=200, batch_size=32, patience=20, verbose=0)
plt.plot(hist.history['loss'], label='train'); plt.plot(hist.history['val_loss'], label='val')
plt.legend(); plt.title('ANN loss'); plt.xlabel('epoch'); plt.show()

## 3. Train the XGBoost meta-head on ANN embeddings + calibrate

In [ ]:
hybrid = models.HybridModel(ann, classes, model_name='cbc_multiclass', version='v1.0.0')
hybrid.fit_meta(Xtr, ytr, Xval, yval, n_estimators=300, max_depth=4)
print('temperature T =', round(hybrid.temperature.T, 3))

## 4. Evaluate on the held-out test set
We report **macro-F1** and **per-class recall** — the evidence that the model is balanced across diseases, not just globally accurate.

In [ ]:
proba = hybrid.predict_proba(Xte)
pred = proba.argmax(1)
metrics = models.compute_metrics(yte, pred, classes)
print('macro-F1          :', metrics['macro_f1'])
print('balanced accuracy :', metrics['balanced_accuracy'])
print('\nper-class:')
for c in classes:
    pc = metrics['per_class'][c]
    print(f'  {c:34} recall={pc["recall"]:.2f} precision={pc["precision"]:.2f} support={pc["support"]}')

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(yte, pred, display_labels=range(n))
plt.title('Model A confusion matrix (test)'); plt.show()

## 5. Confidence-score output (what the Doctor Portal shows)
No categorical diagnosis — just calibrated confidences the doctor then confirms.

In [ ]:
scores = hybrid.predict_scores(Xte[:3])
for i, s in enumerate(scores):
    top = sorted(s.items(), key=lambda kv: -kv[1])[:4]
    print(f'sample {i}:', {k: round(v,3) for k,v in top}, '...')

## 5b. Thalassemia-trait screening flag (rule layer, runs alongside)
The interpretable Mentzer/discriminant layer (`fedhealth_ml.rules`) runs on the same CBC indices and surfaces *possible beta-thalassemia trait* next to the ML scores. It only fires on microcytic anemia (clinical gate) and screens — it does not diagnose.

In [ ]:
from fedhealth_ml import rules
# Show the flag for the same 3 samples, using their raw (unscaled) CBC.
raw_rows = sp['X_test'].iloc[:3].to_dict('records') if 'sp' in dir() else None
# If sp isn't in memory (ran 03 standalone), reload raw test rows:
if raw_rows is None:
    X_raw, y_raw = pp.load_anemia(ANEMIA)
    raw_rows = X_raw.iloc[:3].to_dict('records')
for i, r in enumerate(raw_rows):
    f = rules.thalassemia_trait_flag(r)
    print(f'sample {i}: applicable={f["applicable"]} flag={f["flag"]} '
          f'mentzer={f["mentzer_index"]} :: {f["rationale"]}')

## 6. Save model + preprocessor + metrics (for the FL client, inference service, and the ml_models registry)

In [ ]:
hybrid.save(ART)
models.write_registry_metrics(ART, metrics)
os.makedirs(DRIVE_ART, exist_ok=True)
for filename in ['ann.keras', 'xgb.json', 'meta.json', 'metrics.json',
                 'preprocessor.joblib', 'arrays.npz']:
    local_path = Path(ART) / filename
    if local_path.exists():
        shutil.copy2(local_path, Path(DRIVE_ART) / filename)
print('saved locally ->', ART)
print('backed up to Drive ->', DRIVE_ART)

## 7. How this plugs into Flower (Module 4) and the registry
```python
# FL client: the ANN weights ARE the federated parameters.
w = models.get_ann_weights(hybrid.ann)   # list[np.ndarray] -> Flower params
# FedAvg averages these across labs; Paillier (Module 5) encrypts them.
# models.set_ann_weights(hybrid.ann, global_weights)  # apply the global model
```
Register the model version by POST-ing `meta.json` + `metrics.json` to the backend's model-registry route (Module 6), which calls `MlModels.register(...)` from Module 1. The `metrics.json` per-class recall is what proves, on the record, that the model isn't disease-biased.